# Results analysis

Reads every `results/<model>/*.csv` and renders the readable version:
class-ordering table, AUC battery (with circularity warnings), ablation
prediction, sensitivity stability, bottleneck crosstab, repair matrix, and a
printed headline summary. Rerun any time new results land.

In [ ]:
import glob, os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)

RESULTS = "results"
def load(pattern):
    out = {}
    for p in sorted(glob.glob(f"{RESULTS}/*/{pattern}")):
        m = p.split(os.sep)[-2]
        if m != "aggregate":
            out[m] = pd.read_csv(p)
    return out

ob = load("obstruction.csv"); aucb = load("auc_battery.csv")
sens = load("sensitivity.csv"); bott = load("bottleneck.csv")
rep = load("repair_matrix.csv")
print("models with obstruction results:", list(ob))

## 1. Class ordering — the core claim
Theory demands median ob: source > transport > selection > correct.

In [ ]:
ORDER = ["source", "transport", "selection", "correct"]
rows = []
for m, df in ob.items():
    r = {"model": m, "n_fail": (df.verdict != "correct").sum()}
    for v in ORDER:
        r[v] = df[df.verdict == v].ob.median()
    vals = [r[v] for v in ORDER if pd.notna(r[v])]
    r["monotone"] = all(a >= b for a, b in zip(vals, vals[1:]))
    rows.append(r)
t1 = pd.DataFrame(rows).set_index("model")
display(t1)

fig, axes = plt.subplots(1, len(ob), figsize=(4.5*len(ob), 3), squeeze=False)
for ax, (m, df) in zip(axes[0], ob.items()):
    data = [df[df.verdict == v].ob.dropna() for v in ORDER]
    ax.boxplot([d for d in data if len(d)],
               labels=[v[:4] for v, d in zip(ORDER, data) if len(d)])
    ax.set_title(m); ax.set_ylabel("R(S)")
plt.tight_layout(); plt.show()

## 2. AUC battery
**Circularity warning:** with stand-in verdicts, `internal` (pi_S) and any
affine feature involving pi_S (ob_norm) partially *define* the labels —
internal = 1.000 exactly is definition leakage, not science. Clean
comparisons with stand-in labels: **affine-vs-black-box**, and raw-ob-only
results. All four tiers become valid with pipeline verdict labels.

In [ ]:
if aucb:
    t2 = pd.concat([df.assign(model=m) for m, df in aucb.items()])
    piv = t2.pivot_table(index="task", columns="model",
                         values=["black-box", "internal", "affine"])
    display(piv.round(3))
    pooled = t2.groupby("task")[["black-box", "internal", "affine"]].mean()
    print("pooled:"); display(pooled.round(3))
    for _, r in pooled.iterrows():
        if r["internal"] > 0.999:
            print(f"  ! {r.name}: internal = 1.000 -> circularity with "
                  f"stand-in labels; ignore until pipeline labels used")

## 3. Does the ledger predict causal ablation effects?
Prediction (corrected sign): ablation severs *delivered* support, ob measures
the delivery *shortfall*, so across a mixed cohort ob should **anti**correlate
with effect size, while delivery surplus (-gap0, shortfall as negative) should
**positively** predict it. attn_mass is the baseline and should be ~0.
Per-class rows test the within-verdict version.

In [ ]:
from scipy.stats import spearmanr
rows = []
for m, df in ob.items():
    f = df[(df.verdict != "correct")].dropna(subset=["ablation_effect"])
    if len(f) < 10: continue
    r = {"model": m, "n": len(f)}
    for feat in ("ob", "attn_mass", "pi_S"):
        rho, p = spearmanr(f[feat], f.ablation_effect)
        r[f"rho_{feat}"] = rho; r[f"p_{feat}"] = p
    rows.append(r)
if rows: display(pd.DataFrame(rows).set_index("model").round(3))
else: print("not enough ablation effects recorded")

## 4. Sensitivity — stability of the instrument
Strong form of the claim: the AUC should be flat across the entire grid even
where the binary verdict split moves (quantile choice).

In [ ]:
for m, df in sens.items():
    print(f"-- {m}: AUC range [{df.auc_pt_vs_sc.min():.3f}, "
          f"{df.auc_pt_vs_sc.max():.3f}]; "
          f"agreement by quantile:")
    display(df.pivot_table(index="pin_frac", columns="quantile",
                           values="agree_with_ref").round(2))

## 5. First-token bottleneck by verdict
(word-level divergence; rerun run_bottleneck.py after the same_word fix if
this table shows div@0 ~ 0.5 for correct rows — that was the token-boundary
artifact)

In [ ]:
for m, df in bott.items():
    d = df[df.diverged == True]  # noqa: E712
    agg = d.groupby("verdict").agg(
        n=("div_step", "size"),
        div0=("div_step", lambda s: (s == 0).mean()),
        rails=("back_on_rails", "mean"),
        med_div=("div_step", "median"))
    print(f"-- {m}"); display(agg.round(2))
    corr_div = (df[df.verdict == "correct"].diverged == True).mean()
    if corr_div > 0.25:
        print(f"  ! {corr_div:.0%} of correct rows 'diverge' -> "
              f"rerun with the word-level same_word fix")

## 6. Repair matrix (pooled)

In [ ]:
if rep:
    t6 = pd.concat([df.assign(model=m) for m, df in rep.items()])
    cols = [c for c in ("source_patch", "transport_edges",
                        "selection_demoters", "random_heads") if c in t6]
    display(t6.groupby("verdict")[cols].agg(["mean", "count"]).round(2))
    fcols = [c + "_flip" for c in cols if c + "_flip" in t6]
    print("flip rates:")
    display(t6.groupby("verdict")[fcols].mean().round(2))

## 7. Headline summary

In [ ]:
print("=" * 60)
for m, df in ob.items():
    f = df[df.verdict != "correct"]
    meds = {v: df[df.verdict == v].ob.median() for v in ORDER}
    vals = [meds[v] for v in ORDER if pd.notna(meds[v])]
    print(f"{m}:")
    print(f"  ordering {' > '.join(v[:4] for v in ORDER if pd.notna(meds[v]))}: "
          f"{'HOLDS' if all(a >= b for a, b in zip(vals, vals[1:])) else 'VIOLATED'}"
          f"  ({', '.join(f'{v:.2f}' for v in vals)})")
    z = (df[df.verdict == 'correct'].ob < 1e-6).mean()
    print(f"  correct with ob = 0: {z:.0%} "
          f"(rest are bottom-decile successes, expected ~10%)")
if sens:
    lo = min(df.auc_pt_vs_sc.min() for df in sens.values())
    hi = max(df.auc_pt_vs_sc.max() for df in sens.values())
    print(f"sensitivity: AUC in [{lo:.3f}, {hi:.3f}] across all configs/models")
print("=" * 60)
print("NEXT: swap stand-in verdicts for pipeline labels; rerun ob+auc stages.")

## Full results dump — print everything, save to Excel/HTML/zip

In [ ]:
# ---------- full results dump: print everything, save nicely ----------
import glob, os, pandas as pd
from datetime import date

OUT = "results/full_report"
os.makedirs(OUT, exist_ok=True)

def collect():
    found = []
    for p in sorted(glob.glob("results/*/*.csv")):
        parts = p.split(os.sep)
        model, stage = parts[1], parts[2].replace(".csv", "")
        if model != "full_report":
            found.append((model, stage, pd.read_csv(p)))
    return found

tables = collect()
print(f"{len(tables)} result tables found")
for model, stage, df in tables:
    print(f"\n{'='*70}\n### {model} — {stage}   ({len(df)} rows)\n{'='*70}")
    with pd.option_context("display.max_rows", None,
                           "display.max_columns", None,
                           "display.width", 250, "display.precision", 4):
        display(df)

xlsx_path = f"{OUT}/all_results.xlsx"
try:
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
        for model, stage, df in tables:
            df.to_excel(xw, sheet_name=f"{model[:18]}_{stage[:11]}"[:31],
                        index=False)
    print(f"Excel workbook -> {xlsx_path}")
except ImportError:
    print("pip install openpyxl for the Excel export")

css = ("<style>body{font-family:sans-serif;margin:2em}"
       "table{border-collapse:collapse;font-size:12px;margin:1em 0}"
       "td,th{border:1px solid #ccc;padding:3px 8px;text-align:right}"
       "th{background:#f0f0f0}h2{border-bottom:2px solid #444;padding-top:1em}"
       "tr:nth-child(even){background:#fafafa}</style>")
parts = [f"<html><head>{css}</head><body>",
         f"<h1>Affine-ledger experiments — full results "
         f"({date.today().isoformat()})</h1>"]
for model, stage, df in tables:
    parts.append(f"<h2>{model} — {stage} ({len(df)} rows)</h2>")
    parts.append(df.to_html(index=False, float_format=lambda v: f"{v:.4f}"))
figs = sorted(glob.glob("results/**/*.png", recursive=True))
if figs:
    parts.append("<h2>Figures</h2>")
    for f in figs:
        rel = os.path.relpath(f, OUT)
        parts.append(f"<h3>{f}</h3><img src='{rel}' style='max-width:900px'>")
parts.append("</body></html>")
open(f"{OUT}/all_results.html", "w").write("\n".join(parts))
print(f"HTML report    -> {OUT}/all_results.html")

import shutil
print("zipped archive ->", shutil.make_archive("results_bundle", "zip", "results"))